# MULTICLASS WITH XGBOOST

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\processed\listings_with_distance_clean.csv')

We start with assigning the y variable to our target

Since we are doing multiclass, we choose subtype as the target to classify

In [3]:
# Check the raw imbalance (This is critical to see)
print("--- Raw Subtype Counts ---")
print(df['subtype'].value_counts())

--- Raw Subtype Counts ---
subtype
Residence         11570
Apartment          7405
Villa              1072
Mixed Building      620
Duplex              523
Penthouse           495
Ground Floor        449
Studio              263
Master House        187
Bungalow            151
Cottage             150
Chalet              125
Loft                 77
Triplex              58
Mansion              24
Name: count, dtype: int64


XGBoost hates strings, we need LabelEncoder to convert str to int

In [4]:
from sklearn.preprocessing import LabelEncoder

#Instantiate and fit the LabelEncoder

le = LabelEncoder()
df['subtype_encoded'] = le.fit_transform(df['subtype'])

Create a dict with the new int labels mapped to the subtype str for later translation

In [5]:
#Save the Translation Dictionary (Crucial for later)
# This creates a dictionary like {0: 'Apartment', 1: 'Castle', ...}

label_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print("--- The Translation Dictionary ---")
for key, value in label_mapping.items():
    print(f"{key}: {value}")

label_mapping

--- The Translation Dictionary ---
0: Apartment
1: Bungalow
2: Chalet
3: Cottage
4: Duplex
5: Ground Floor
6: Loft
7: Mansion
8: Master House
9: Mixed Building
10: Penthouse
11: Residence
12: Studio
13: Triplex
14: Villa


{np.int64(0): 'Apartment',
 np.int64(1): 'Bungalow',
 np.int64(2): 'Chalet',
 np.int64(3): 'Cottage',
 np.int64(4): 'Duplex',
 np.int64(5): 'Ground Floor',
 np.int64(6): 'Loft',
 np.int64(7): 'Mansion',
 np.int64(8): 'Master House',
 np.int64(9): 'Mixed Building',
 np.int64(10): 'Penthouse',
 np.int64(11): 'Residence',
 np.int64(12): 'Studio',
 np.int64(13): 'Triplex',
 np.int64(14): 'Villa'}

In [6]:
df.columns

Index(['locality', 'region', 'zip_code', 'property_type', 'subtype',
       'price_eur', 'type_of_sale', 'num_rooms', 'living_area_m2',
       'fully_equipped_kitchen', 'furnished', 'terrace', 'terrace_area_m2',
       'garden', 'garden_area_m2', 'land_surface_m2', 'num_facades',
       'swimming_pool', 'state_of_building', 'num_bathrooms', 'dist_train_km',
       'dist_bus_km', 'subtype_encoded'],
      dtype='str')

Define X and y vars

In [7]:
cols_to_drop = [
    'property_type', 
    'locality',
    'region',
    'zip_code',
    'subtype',
    'type_of_sale',
    'state_of_building',
    'subtype_encoded'
]

X = df.drop(columns=cols_to_drop)
y = df['subtype_encoded']

print(f"{X.shape=}")
print(f"{y.shape=}")

X.shape=(23169, 15)
y.shape=(23169,)


In [8]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

#Split the data
# Stratify ensures we don't accidentally put all 24 Mansions into the test set!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


The Weight Balancing Trick

Because we have 24 Mansions and 11,570 Residences, we have to artificially balance the scales. We do this by calculating Sample Weights. We will tell XGBoost: "If you get a Residence wrong, minus 1 point. If you get a Mansion wrong, minus 480 points."

In [9]:
#Compute Sample Weights (The Imbalance Fix)
#This calculates the exact mathematical penalty for all 15 classes based on their rarity.
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


In [10]:
#Instantiate the XGBoost Multi-Class Engine
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=15,
    eval_metric='mlogloss',
    random_state=42
)

In [11]:
print("Training XGBoost Relay Race... (This might take a few seconds)")
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

#Predict on the unseen Test data
y_pred = xgb_model.predict(X_test)

Training XGBoost Relay Race... (This might take a few seconds)


In [12]:
# Print the Classification Report using our saved Translation Dictionary!
# We pull the names in the exact order (0 to 14) so the report is human-readable.
target_names = [label_mapping[i] for i in range(15)]
print(classification_report(y_test, y_pred, target_names=target_names))


                precision    recall  f1-score   support

     Apartment       0.87      0.81      0.84      1481
      Bungalow       0.23      0.30      0.26        30
        Chalet       0.46      0.64      0.53        25
       Cottage       0.24      0.30      0.26        30
        Duplex       0.24      0.37      0.29       105
  Ground Floor       0.30      0.34      0.32        90
          Loft       0.50      0.33      0.40        15
       Mansion       0.14      0.20      0.17         5
  Master House       0.20      0.19      0.19        37
Mixed Building       0.35      0.53      0.42       124
     Penthouse       0.36      0.53      0.42        99
     Residence       0.92      0.81      0.86      2314
        Studio       0.63      0.77      0.69        53
       Triplex       0.25      0.17      0.20        12
         Villa       0.36      0.64      0.46       214

      accuracy                           0.75      4634
     macro avg       0.40      0.46      0.42 

Low scores...

XGBoost is the most powerful tabular math engine in the world, but math cannot solve human subjectivity. 

Think about your 15 categories. What is the actual, strict physical difference between a "Villa", a "Mansion", and a "Master House"?
If a real estate agent is selling a 300m² house with a big garden: 

Agent A might call it a Villa. 

Agent B might call it a Mansion to make it sound fancier. 

Agent C might just label it a Residence.

If the humans creating the data are inconsistent and subjective, the algorithm cannot find a perfect mathematical rule. 

XGBoost is currently hitting the "Information Ceiling" of human language.

We can still squeeze out a bit more performance by tuning the hyperparams like eta, max_depth and subsample

In [13]:
from sklearn.inspection import permutation_importance

XGBC_perm_result = permutation_importance(xgb_model, 
                                         X_test, 
                                         y_test, 
                                         scoring='f1_macro', 
                                         n_repeats=10, 
                                         random_state=42)

XGBC_perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': XGBC_perm_result.importances_mean,
    'std_dev': XGBC_perm_result.importances_std
}).sort_values(by='importance', ascending=False)

XGBC_perm_df

,feature,importance,std_dev
9,land_surface_m2,0.231176,0.013652
2,living_area_m2,0.151914,0.007238
0,price_eur,0.072209,0.014983
10,num_facades,0.050461,0.008369
6,terrace_area_m2,0.046919,0.007925
13,dist_train_km,0.041493,0.009630
14,dist_bus_km,0.036164,0.008518
8,garden_area_m2,0.027142,0.005855
1,num_rooms,0.025443,0.008113
12,num_bathrooms,0.017019,0.005345


### 📈 XGBoost Model Progress Tracker
#### Goal: Subtype Multiclassification

| Iteration | Key Modifications | Train [score metric] | Test [score metric] | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** |  |  |  |  |  |
| **2** |  |  |  |  |  |
| **3** |  |  |  |  |  |

> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far: 